# Grid epidemic (SIR) with columns

An SIR epidemic on a Moore grid. `state` and `days_infected` are `Column`s, so the three model reporters are numpy comparisons and the data collector slices the column table. The per-agent work is the move and the infection loop over cellmates.

In [1]:
import json
import time

import pandas as pd


def bench(label, make_model, steps, agents_of=lambda m: len(m.agents)):
    """Build the model, time ``steps`` steps, print a RESULT line, return a row."""
    model = make_model()
    n = agents_of(model)
    t0 = time.perf_counter()
    model.run_for(steps)
    per_step = (time.perf_counter() - t0) / steps
    row = {
        "label": label,
        "agents": n,
        "ms_per_step": round(per_step * 1e3, 2),
        "us_per_agent_step": round(per_step / max(n, 1) * 1e6, 2),
    }
    print("RESULT " + json.dumps(row))
    return row

In [2]:
from modelly import Column, Model
from modelly.datacollection import DataCollector
from modelly.discrete_space import CellAgent, OrthogonalMooreGrid
from modelly.experimental.scenarios import Scenario

S, I, R = 0, 1, 2


class EpidemicScenario(Scenario):
    n: int = 2000
    side: int = 60
    infection_chance: float = 0.3
    recovery_time: int = 10
    initially_infected: int = 10


class Person(CellAgent):
    state = Column(int, default=S)
    days_infected = Column(int, default=0)

    def __init__(self, model, cell, infected=False):
        super().__init__(model)
        self.cell = cell
        if infected:
            self.state = I

    def step(self):
        self.cell = self.cell.neighborhood.select_random_cell()
        if self.state != I:
            return
        chance = self.scenario.infection_chance
        for other in self.cell.agents:
            if other.state == S and self.random.random() < chance:
                other.state = I
        self.days_infected += 1
        if self.days_infected >= self.scenario.recovery_time:
            self.state = R


class Epidemic(Model):
    def __init__(self, scenario=EpidemicScenario):
        super().__init__(scenario=scenario)
        self.grid = OrthogonalMooreGrid((scenario.side, scenario.side), torus=True, random=self.random)
        cells = self.random.choices(self.grid.all_cells.cells, k=scenario.n)
        infected = [i < scenario.initially_infected for i in range(scenario.n)]
        Person.create_agents(self, scenario.n, cells, infected=infected)
        self.datacollector = DataCollector(
            model_reporters={"S": lambda m: m.count(S), "I": lambda m: m.count(I), "R": lambda m: m.count(R)},
            agent_reporters={"state": "state"},
        )

    def count(self, state):
        return int((self.columns[Person].state == state).sum())

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)

In [3]:
rows = []
rows.append(bench("2k agents, 60x60", lambda: Epidemic(EpidemicScenario(n=2000, side=60, rng=1)), steps=20))
rows.append(bench("8k agents, 120x120", lambda: Epidemic(EpidemicScenario(n=8000, side=120, rng=1)), steps=20))
rows.append(bench("32k agents, 240x240", lambda: Epidemic(EpidemicScenario(n=32000, side=240, rng=1)), steps=10))

RESULT {"label": "2k agents, 60x60", "agents": 2000, "ms_per_step": 6.41, "us_per_agent_step": 3.2}


RESULT {"label": "8k agents, 120x120", "agents": 8000, "ms_per_step": 30.09, "us_per_agent_step": 3.76}


RESULT {"label": "32k agents, 240x240", "agents": 32000, "ms_per_step": 165.63, "us_per_agent_step": 5.18}


In [4]:
pd.DataFrame(rows).set_index('label')

,agents,ms_per_step,us_per_agent_step
label,,,
"2k agents, 60x60",2000,6.41,3.20
"8k agents, 120x120",8000,30.09,3.76
"32k agents, 240x240",32000,165.63,5.18
